# 64×64 Streaming 2D Convolution on PYNQ — Float Version

This notebook tests the `conv2d_stream` HLS accelerator for a **64×64 image** and a **3×3 kernel** using a `float` / `np.float32` data path.

The accelerator performs full convolution using a streaming AXI-DMA input/output path. Kernel coefficients are written once through AXI-Lite registers before streaming begins.

Expected dimensions:

```text
Input image      : 64 × 64
Kernel           : 3 × 3
Output image     : 66 × 66
DMA input words  : 4224 = 66 × 64 = 4096 real pixels + 128 trailing zeros
DMA output words : 4356 = 66 × 66
Data format      : float32 / IEEE-754 single precision
```

The HLS implementation assumes the kernel is already in the sliding-window orientation. If mathematical convolution is desired, provide the already-flipped kernel.

Important: the DMA buffers use `np.float32`. The AXI-Lite kernel registers are written using raw IEEE-754 32-bit bit patterns.


## 1. Imports and overlay loading

Place `conv2d_design.bit` and `conv2d_design.hwh` in the same directory as this notebook before running it.


In [1]:
import time
import struct
import numpy as np
from pynq import Overlay, allocate


In [2]:
ol = Overlay("conv2d-float.bit")
print("Overlay loaded successfully")
print("IP blocks:")
for name in ol.ip_dict.keys():
    print("  ", name)

# Expected handles from the Vivado block design
dma = ol.axi_dma_0
conv2d = ol.conv2d_stream_0
hw_timer = ol.axi_timer_0


Overlay loaded successfully
IP blocks:
   conv2d_stream_0
   axi_timer_0
   axi_dma_0
   ps7


## 2. Dimensions

These constants must match the HLS header file.


In [3]:
IH = 64
IW = 64
KH = 3
KW = 3

OH = IH + KH - 1   # 66
OW = IW + KW - 1   # 66

N_INPUT_REAL = IH * IW          # 4096 real image pixels
N_INPUT_STREAM = OH * IW        # 66 * 64 = 4224, because current HLS reads two extra zero rows
N_OUTPUT = OH * OW              # 66 * 66 = 4356

print(f"Input image shape     : {IH} x {IW}")
print(f"Kernel shape          : {KH} x {KW}")
print(f"Output image shape    : {OH} x {OW}")
print(f"DMA input words       : {N_INPUT_STREAM}")
print(f"DMA output words      : {N_OUTPUT}")
print("Data type             : float32")


Input image shape     : 64 x 64
Kernel shape          : 3 x 3
Output image shape    : 66 x 66
DMA input words       : 4224
DMA output words      : 4356
Data type             : float32


## 3. AXI-Lite register map

These offsets are the typical Vitis HLS AXI-Lite register offsets for this function signature. Confirm against the generated HLS driver header if needed:

```bash
grep -n "k00\|k01\|k02\|k10\|k11\|k12\|k20\|k21\|k22" \
  hls_2dconv_stream_pipelined/sol1/impl/ip/drivers/conv2d_stream_v1_0/src/xconv2d_stream_hw.h
```

For `float` AXI-Lite arguments, write the raw IEEE-754 bit pattern, not the Python float object directly.


In [4]:
CTRL_REG = 0x00

K_OFFSETS = {
    "k00": 0x10,
    "k01": 0x18,
    "k02": 0x20,
    "k10": 0x28,
    "k11": 0x30,
    "k12": 0x38,
    "k20": 0x40,
    "k21": 0x48,
    "k22": 0x50,
}

def float32_to_u32(x):
    """Pack a Python/NumPy float into raw IEEE-754 uint32 bits."""
    return struct.unpack("<I", struct.pack("<f", np.float32(x)))[0]

def u32_to_float32(x):
    """Unpack raw IEEE-754 uint32 bits into a NumPy float32."""
    return np.float32(struct.unpack("<f", struct.pack("<I", int(x) & 0xFFFFFFFF))[0])

def write_kernel(ip, kernel):
    """Write a 3x3 float32 kernel to AXI-Lite registers."""
    kernel = np.asarray(kernel, dtype=np.float32)
    assert kernel.shape == (3, 3)

    ip.write(K_OFFSETS["k00"], float32_to_u32(kernel[0, 0]))
    ip.write(K_OFFSETS["k01"], float32_to_u32(kernel[0, 1]))
    ip.write(K_OFFSETS["k02"], float32_to_u32(kernel[0, 2]))

    ip.write(K_OFFSETS["k10"], float32_to_u32(kernel[1, 0]))
    ip.write(K_OFFSETS["k11"], float32_to_u32(kernel[1, 1]))
    ip.write(K_OFFSETS["k12"], float32_to_u32(kernel[1, 2]))

    ip.write(K_OFFSETS["k20"], float32_to_u32(kernel[2, 0]))
    ip.write(K_OFFSETS["k21"], float32_to_u32(kernel[2, 1]))
    ip.write(K_OFFSETS["k22"], float32_to_u32(kernel[2, 2]))

def start_ip(ip):
    """Start HLS IP."""
    ip.write(CTRL_REG, 0x01)

def wait_ip_done(ip, timeout_s=1.0):
    """Optional polling of ap_done. DMA wait is usually enough."""
    t0 = time.perf_counter()
    while (ip.read(CTRL_REG) & 0x02) == 0:
        if time.perf_counter() - t0 > timeout_s:
            raise TimeoutError("conv2d_stream did not assert ap_done")
    return True


## 4. Optimized software reference model

This matches the HLS implementation: the kernel is already in the orientation used by the sliding-window hardware, so the reference does **not** flip the kernel again.

The notebook tries to use `scipy.signal.correlate2d`. If SciPy is unavailable, it falls back to a vectorized NumPy sliding-window implementation.


In [5]:
try:
    from scipy.signal import correlate2d
    SCIPY_AVAILABLE = True
except Exception as e:
    SCIPY_AVAILABLE = False
    print("SciPy unavailable; using NumPy sliding-window fallback.")
    print("Reason:", repr(e))

def conv2d_reference_float32(image, kernel):
    image = np.asarray(image, dtype=np.float32)
    kernel = np.asarray(kernel, dtype=np.float32)

    if SCIPY_AVAILABLE:
        # correlate2d matches the HLS convention: no extra kernel flip.
        return correlate2d(image, kernel, mode="full", boundary="fill", fillvalue=0).astype(np.float32)

    from numpy.lib.stride_tricks import sliding_window_view

    ih, iw = image.shape
    kh, kw = kernel.shape
    oh = ih + kh - 1
    ow = iw + kw - 1

    padded = np.zeros((oh + 2, ow + 2), dtype=np.float32)
    padded[2:2+ih, 2:2+iw] = image

    windows = sliding_window_view(padded, (kh, kw))
    return np.sum(windows * kernel, axis=(2, 3), dtype=np.float32)


## 5. Prepare input image, kernel, and expected output

To keep the notebook readable, only small slices of the 64×64 image and 66×66 output are printed.


In [6]:
# Deterministic 64x64 test image.
# Keep values moderate for clear float32 behavior.
image = (np.arange(1, IH * IW + 1, dtype=np.float32).reshape(IH, IW) % 256).astype(np.float32)

# Kernel loaded into HLS. This is assumed to be in sliding-window orientation.
kernel = np.array([
    [-1.0, -2.0, -3.0],
    [-4.0, -5.0, -6.0],
    [-7.0, -8.0, -9.0],
], dtype=np.float32)

expected = conv2d_reference_float32(image, kernel)

print("Input image shape:", image.shape)
print("Top-left 5x5 of input image:")
print(image[:5, :5])

print("Kernel loaded into HLS:")
print(kernel)

print("Expected output shape:", expected.shape)
print("Top-left 8x8 of expected output:")
print(expected[:8, :8])
print("Expected output range:", float(expected.min()), "to", float(expected.max()))


Input image shape: (64, 64)
Top-left 5x5 of input image:
[[  1.   2.   3.   4.   5.]
 [ 65.  66.  67.  68.  69.]
 [129. 130. 131. 132. 133.]
 [193. 194. 195. 196. 197.]
 [  1.   2.   3.   4.   5.]]
Kernel loaded into HLS:
[[-1. -2. -3.]
 [-4. -5. -6.]
 [-7. -8. -9.]]
Expected output shape: (66, 66)
Top-left 8x8 of expected output:
[[   -9.   -26.   -50.   -74.   -98.  -122.  -146.  -170.]
 [ -591. -1131. -1618. -1657. -1696. -1735. -1774. -1813.]
 [-1554. -2931. -4128. -4173. -4218. -4263. -4308. -4353.]
 [-2706. -5043. -7008. -7053. -7098. -7143. -7188. -7233.]
 [-1554. -2803. -3744. -3789. -3834. -3879. -3924. -3969.]
 [-1170. -2099. -2784. -2829. -2874. -2919. -2964. -3009.]
 [-1554. -2931. -4128. -4173. -4218. -4263. -4308. -4353.]
 [-2706. -5043. -7008. -7053. -7098. -7143. -7188. -7233.]]
Expected output range: -9708.0 to 0.0


## 6. Allocate DMA buffers

The current HLS implementation reads `OH × IW = 4224` input words. The first 4096 words are the real image pixels, and the final 128 words are zeros representing two extra zero rows.

For the float version, the DMA buffers use `np.float32`.


In [7]:
in_buf = allocate(shape=(N_INPUT_STREAM,), dtype=np.float32)
out_buf = allocate(shape=(N_OUTPUT,), dtype=np.float32)

in_buf[:N_INPUT_REAL] = image.flatten()
in_buf[N_INPUT_REAL:] = np.float32(0.0)
out_buf[:] = np.float32(0.0)

print(f"Input buffer shape : {in_buf.shape}")
print(f"Output buffer shape: {out_buf.shape}")
print("First 8 input words:", np.array(in_buf[:8]))
print("Last 8 input words, should be zeros:", np.array(in_buf[-8:]))


Input buffer shape : (4224,)
Output buffer shape: (4356,)
First 8 input words: [1. 2. 3. 4. 5. 6. 7. 8.]
Last 8 input words, should be zeros: [0. 0. 0. 0. 0. 0. 0. 0.]


## 7. Run the accelerator using AXI DMA

Safe order:

1. Program kernel coefficients over AXI-Lite
2. Start DMA receive channel
3. Start DMA send channel
4. Start the HLS IP
5. Wait for both DMA channels


In [8]:
print("--- DMA Streaming 2D Convolution, float32 ---")

write_kernel(conv2d, kernel)

t_start = time.perf_counter()

dma.recvchannel.transfer(out_buf)
dma.sendchannel.transfer(in_buf)
start_ip(conv2d)

dma.sendchannel.wait()
dma.recvchannel.wait()

t_dma = time.perf_counter() - t_start

hw_out = np.array(out_buf, dtype=np.float32).reshape(OH, OW)

print(f"DMA transfer + accelerator time: {t_dma * 1e3:.3f} ms")
print("Hardware output shape:", hw_out.shape)
print("Top-left 8x8 of hardware output:")
print(hw_out[:8, :8])


--- DMA Streaming 2D Convolution, float32 ---
DMA transfer + accelerator time: 7.995 ms
Hardware output shape: (66, 66)
Top-left 8x8 of hardware output:
[[   -9.   -26.   -50.   -74.   -98.  -122.  -146.  -170.]
 [ -591. -1131. -1618. -1657. -1696. -1735. -1774. -1813.]
 [-1554. -2931. -4128. -4173. -4218. -4263. -4308. -4353.]
 [-2706. -5043. -7008. -7053. -7098. -7143. -7188. -7233.]
 [-1554. -2803. -3744. -3789. -3834. -3879. -3924. -3969.]
 [-1170. -2099. -2784. -2829. -2874. -2919. -2964. -3009.]
 [-1554. -2931. -4128. -4173. -4218. -4263. -4308. -4353.]
 [-2706. -5043. -7008. -7053. -7098. -7143. -7188. -7233.]]


## 8. Verify correctness

The hardware output should match the optimized software reference within a small float tolerance.


In [9]:
diff = hw_out - expected
max_abs_diff = np.max(np.abs(diff))

print("Difference shape:", diff.shape)
print("Top-left 8x8 of difference:")
print(diff[:8, :8])

print(f"Max absolute difference = {max_abs_diff}")

TOL = 1e-3
if max_abs_diff <= TOL:
    print(f"PASS: hardware output matches software reference within tolerance {TOL}")
else:
    print(f"FAIL: hardware output differs from software reference beyond tolerance {TOL}")


Difference shape: (66, 66)
Top-left 8x8 of difference:
[[0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]]
Max absolute difference = 0.0
PASS: hardware output matches software reference within tolerance 0.001


## 9. AXI Timer measurement

This section reruns the accelerator and measures elapsed cycles using the AXI Timer IP.


In [10]:
TCSR0 = 0x00
TLR0  = 0x04
TCR0  = 0x08

def timer_start(timer):
    timer.write(TLR0, 0)
    timer.write(TCSR0, 0x020)   # LOAD
    timer.write(TCSR0, 0x080)   # ENT, count up

def timer_stop(timer):
    cycles = timer.read(TCR0)
    timer.write(TCSR0, 0x000)
    return cycles

# Reinitialize buffers
in_buf[:N_INPUT_REAL] = image.flatten()
in_buf[N_INPUT_REAL:] = np.float32(0.0)
out_buf[:] = np.float32(0.0)

write_kernel(conv2d, kernel)

dma.recvchannel.transfer(out_buf)
dma.sendchannel.transfer(in_buf)

timer_start(hw_timer)
start_ip(conv2d)

dma.sendchannel.wait()
dma.recvchannel.wait()

cycles = timer_stop(hw_timer)

FCLK_MHZ = 100.0
hw_time_us = cycles / FCLK_MHZ

print(f"HW timer: {cycles} cycles = {hw_time_us:.3f} us @ {FCLK_MHZ:.0f} MHz")

output_pixels = OH * OW
pixels_per_cycle = output_pixels / cycles
cycles_per_pixel = cycles / output_pixels
mpixels_per_sec = pixels_per_cycle * FCLK_MHZ

print(f"Board-level throughput: {pixels_per_cycle:.6f} output pixels/cycle")
print(f"Board-level cost      : {cycles_per_pixel:.3f} cycles/output pixel")
print(f"Board-level rate      : {mpixels_per_sec:.3f} MPixels/s")


HW timer: 128270 cycles = 1282.700 us @ 100 MHz
Board-level throughput: 0.033960 output pixels/cycle
Board-level cost      : 29.447 cycles/output pixel
Board-level rate      : 3.396 MPixels/s


# Scipy Comparison

In [11]:
# ------------------------------------------------------------
# SciPy correlate2d timing baseline for float version
# ------------------------------------------------------------
import time
import numpy as np

try:
    from scipy.signal import correlate2d
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False

print("\n--- SciPy correlate2d Timing Baseline ---")

if not SCIPY_AVAILABLE:
    print("SciPy is not available on this PYNQ image.")
    print("Install scipy or use the NumPy sliding-window fallback instead.")
else:
    image_float = image.astype(np.float32)
    kernel_float = kernel.astype(np.float32)

    # HLS convention:
    # The kernel is already in sliding-window orientation, so use correlate2d,
    # not convolve2d.
    y_scipy = correlate2d(
        image_float,
        kernel_float,
        mode="full",
        boundary="fill",
        fillvalue=0
    ).astype(np.float32)

    # hw_out should be the FPGA output reshaped as OH x OW.
    # If your notebook names it differently, replace hw_out below.
    max_diff_scipy = np.max(np.abs(hw_out - y_scipy))

    print(f"Max |FPGA float output - SciPy correlate2d| = {max_diff_scipy:.8f}")

    FLOAT_TOL = 1e-3

    if max_diff_scipy <= FLOAT_TOL:
        print(f"PASS: FPGA output matches SciPy within tolerance {FLOAT_TOL}")
    else:
        print(f"WARNING: difference exceeds tolerance {FLOAT_TOL}")

    # Timing
    N_RUNS = 1000

    # Warm-up
    _ = correlate2d(
        image_float,
        kernel_float,
        mode="full",
        boundary="fill",
        fillvalue=0
    )

    t_start = time.perf_counter()

    for _ in range(N_RUNS):
        y_scipy = correlate2d(
            image_float,
            kernel_float,
            mode="full",
            boundary="fill",
            fillvalue=0
        )

    t_total = time.perf_counter() - t_start
    t_one = t_total / N_RUNS

    print(f"Runs: {N_RUNS}")
    print(f"SciPy correlate2d average time: {t_one * 1e3:.6f} ms")

    print("\n--- FPGA float vs SciPy Timing ---")
    print(f"FPGA DMA+accelerator time: {hw_time_us / 1000:.6f} ms")
    print(f"SciPy correlate2d time:    {t_one * 1e3:.6f} ms")

    speedup = (t_one * 1e6) / hw_time_us
    print(f"FPGA speedup vs SciPy:     {speedup:.3f}x")

    # Throughput
    output_pixels = OH * OW
    pixels_per_cycle = output_pixels / cycles
    cycles_per_pixel = cycles / output_pixels
    mpixels_per_sec = pixels_per_cycle * 100e6 / 1e6

    print("\n--- FPGA Throughput ---")
    print(f"Output pixels:             {output_pixels}")
    print(f"Measured cycles:           {cycles}")
    print(f"Pixels/cycle:              {pixels_per_cycle:.6f}")
    print(f"Cycles/output pixel:       {cycles_per_pixel:.3f}")
    print(f"Throughput @ 100 MHz:      {mpixels_per_sec:.3f} MPixels/s")


--- SciPy correlate2d Timing Baseline ---
Max |FPGA float output - SciPy correlate2d| = 0.00000000
PASS: FPGA output matches SciPy within tolerance 0.001
Runs: 1000
SciPy correlate2d average time: 2.235804 ms

--- FPGA float vs SciPy Timing ---
FPGA DMA+accelerator time: 1.282700 ms
SciPy correlate2d time:    2.235804 ms
FPGA speedup vs SciPy:     1.743x

--- FPGA Throughput ---
Output pixels:             4356
Measured cycles:           128270
Pixels/cycle:              0.033960
Cycles/output pixel:       29.447
Throughput @ 100 MHz:      3.396 MPixels/s


## 10. Optional: optimized software timing baseline

This times the same optimized reference function used for correctness. For the HLS convention, this is correlation-style sliding-window evaluation, not kernel-flipped mathematical convolution.


In [12]:
print("--- Optimized Software Timing Baseline ---")

N_RUNS = 1000

# Warm-up
_ = conv2d_reference_float32(image, kernel)

t_start = time.perf_counter()
for _ in range(N_RUNS):
    y_sw = conv2d_reference_float32(image, kernel)
t_total = time.perf_counter() - t_start

t_one = t_total / N_RUNS
max_diff_sw = np.max(np.abs(y_sw - expected))

print(f"Runs: {N_RUNS}")
print(f"Average optimized software time: {t_one * 1e3:.6f} ms per convolution")
print(f"Max |software timed output - expected| = {max_diff_sw}")

print("\n--- FPGA vs Optimized Software Timing ---")
print(f"FPGA DMA+accelerator time: {hw_time_us / 1000:.6f} ms")
print(f"Optimized software time:   {t_one * 1e3:.6f} ms")

speedup = (t_one * 1e6) / hw_time_us
print(f"FPGA speedup vs software:  {speedup:.3f}x")


--- Optimized Software Timing Baseline ---
Runs: 1000
Average optimized software time: 2.294070 ms per convolution
Max |software timed output - expected| = 0.0

--- FPGA vs Optimized Software Timing ---
FPGA DMA+accelerator time: 1.282700 ms
Optimized software time:   2.294070 ms
FPGA speedup vs software:  1.788x


## 11. Final verification after timed run


In [13]:
hw_out_timer = np.array(out_buf, dtype=np.float32).reshape(OH, OW)
max_abs_diff_timer = np.max(np.abs(hw_out_timer - expected))

print(f"Max absolute difference after timed run = {max_abs_diff_timer}")
if max_abs_diff_timer <= TOL:
    print(f"PASS: timed-run hardware output matches software reference within tolerance {TOL}")
else:
    print(f"FAIL: timed-run hardware output differs from software reference beyond tolerance {TOL}")


Max absolute difference after timed run = 0.0
PASS: timed-run hardware output matches software reference within tolerance 0.001


## 12. Cleanup

Run this cell when finished with the DMA buffers.


In [ ]:
in_buf.freebuffer()
out_buf.freebuffer()
print("DMA buffers freed")
